# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and display key information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print("Description:")
print(metadata.description)

print("\nPublished on:", metadata.date_published)
print("Keywords:", metadata.keywords)
print("\nCite as:\n", metadata.cite_as)


## 2. Data Overview
List available record sets, their `@id`s, and the fields within each set. All entities are referenced by their `@id`.

> Note: For this dataset, actual record set IDs and fields are read from the metadata.

In [ ]:
# Show all available record sets and their field (column) ids
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        print(f"RecordSet: {getattr(record_set, '@id', None)}  Name: {getattr(record_set, 'name', None)}")
        record_set_ids.append(getattr(record_set, '@id', None))
        if hasattr(record_set, 'fields'):
            print("  Fields in this record set:")
            for field in record_set.fields:
                print(f"    Field @id: {getattr(field, '@id', None)} | Column: {getattr(field, 'column', None)} | Name: {getattr(field, 'name', None)}")
        print()
else:
    print("No record sets found in the Croissant schema.")
    # In this dataset, try legacy or typical structures
    record_sets = getattr(metadata, 'record_set', [])  # singular or plural depending on schema
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set.get('@id', None)}")
        record_set_ids.append(record_set.get('@id', None))
        if 'field' in record_set:
            print("  Fields in this record set:")
            for field in record_set['field']:
                print(f"    Field @id: {field.get('@id', None)} | Column: {field.get('column', None)} | Name: {field.get('name', None)}")
        print()
# If no record sets found, inform the user
if not record_set_ids:
    print("No record sets available for extraction in this dataset. Please check the Croissant schema definition.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using its `@id`.

> Note: Update the `record_sets` variable below with the actual record set `@id`s discovered above. For this dataset, we try to programmatically find a record set to continue the example.

In [ ]:
# Collect all available record set ids for loading data
from collections.abc import Iterable

# Attempt to gather record set @id(s) properly
record_sets = []
record_sets_found = False
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        rs_id = getattr(rs, '@id', None)
        if rs_id:
            record_sets.append(rs_id)
    record_sets_found = True
elif hasattr(metadata, 'record_set') and metadata.record_set:
    # Sometimes it's called record_set
    for rs in metadata.record_set:
        rs_id = rs.get('@id', None)
        if rs_id:
            record_sets.append(rs_id)
    record_sets_found = True
else:
    print("No record set definitions found.")

# For this particular dataset, if no record sets are found, try to discover them from the file structure
if not record_sets:
    # List all data files to see if one contains records
    if hasattr(metadata, 'distribution'):
        print("Dataset has distributions (data files):")
        for dist in metadata.distribution:
            print(f"- Distribution @id: {getattr(dist, '@id', dist)}")
    print("Proceeding with dataset.records() with no record set specified, attempting extraction...")

# Load the dataframes for each record set
dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records for Record Set: {record_set}")
        print("Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load data for record set {record_set}: {e}")

# If no specific record sets, try to load without specifying
if not dataframes:
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df
        print(f"Loaded {len(df)} records in default method.")
        print("Columns:", df.columns.tolist())
    except Exception as e:
        print("Could not load records from dataset:", e)

# Inspect the first few rows from the first dataframe
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nSample records for record set '{first_rs}':")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
### Example: Filtering, normalizing, and grouping numeric fields

Let us select a numeric field and conduct simple EDA steps (filtering, normalization, and grouping).

> Update the variable `numeric_field` and `group_field` below with available column names from the loaded dataframe, referencing their `@id` where possible.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Use the first dataframe loaded for demonstration
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Examining record set: {record_set_id}")
    # Try to auto-detect a numeric field
    candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if candidates:
        numeric_field = candidates[0]  # Use first numeric field found
        print(f"Selected numeric field for EDA: {numeric_field}")
    else:
        numeric_field = None
        print("No numeric field available for EDA.")

    # Try to find a grouping field (categorical or string)
    cat_candidates = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < len(df)//2]
    group_field = cat_candidates[0] if cat_candidates else None
    if group_field:
        print(f"Selected group field for grouping: {group_field}")
    else:
        print("No suitable group field found.")

    # Continue EDA only if numeric_field exists
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No records loaded for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship to a group if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found; cannot visualize.")

## 6. Conclusion
In this notebook, you explored the FAIR^2 dataset for mass spectrometry analysis of extracellular vesicles using `mlcroissant`. You loaded the metadata, inspected available record sets and fields by their `@id`, extracted records as DataFrames, and applied basic exploratory data analysis. The visualizations provided a sense of the numeric field distributions and potential group relationships.

Further analysis can include protein function clustering, detailed normalization, or integration with UniProt for annotation enhancement. Consult the Croissant schema or the [dataset's FAIR2 landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more details on available data.